In [1]:
# part 1: 导入相关的 package
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(1024)

In [2]:
import tiktoken
tiktoken.__version__

'0.12.0'

In [3]:
@dataclass
class GPTConfig:
    max_seq_len:int = 512 # 文本最大长度
    batch_size:int = 12
    n_layer:int = 6 # GPT-2中有多少个decoder层
    n_head:int = 12 # 多头注意力中头的个数
    hidden_dim:int = 768
    head_dim:int = hidden_dim // n_head
    dropout:float = 0.1
    vocab_size:int = 50257


## 模型结构
### 1.单头注意力机制

In [4]:
class SingleHeadAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.head_dim = config.head_dim
        self.hidden_dim = config.hidden_dim
        self.n_head = self.hidden_dim // self.head_dim

        self.q = nn.Linear(config.hidden_dim,config.head_dim)
        self.k = nn.Linear(config.hidden_dim,config.head_dim)
        self.v = nn.Linear(config.hidden_dim,config.head_dim)

        # attention_mask只是一个规则模板，不是parameter，不会被optimizer更新，也不会参与反向传播
        # register_buffer会可以将参数注册成模型的缓冲区，后面model.to(device)会自动跟随着走
        self.register_buffer(
            'attention_mask',
            torch.tril(
                torch.ones(config.max_seq_len,config.max_seq_len)
            )
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self,x):
        batch_size,seq_len,_ = x.size()

        #1.计算q/k/v
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        #2.计算attention_weight
        attention_score = q @ k.transpose(-1,-2) / math.sqrt(self.head_dim)
        # 添加attention_mask
        attention_score = attention_score.masked_fill(
            self.attention_mask,
            float('-inf')
        )
        weight = F.softmax(attention_score,dim=-1)
        weight = self.dropout(weight)
        out = weight @ v
        return out


### 2.多头注意力

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head = nn.ModuleList(
            [
                SingleHeadAttention(config) for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.hidden_dim,config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self,x):
        output = torch.cat(
            [h(x) for h in self.head],
            dim = -1
        )

        output = self.proj(output)
        output = self.dropout(output)
        return output

### 3. FFN

In [6]:
class FeedForward(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.hidden_dim,config.hidden_dim*4),
            nn.GELU(),
            nn.Linear(config.hidden_dim*4,config.hidden_dim),
            nn.Dropout(config.dropout)
        )
    
    def forward(self,x):
        return self.net(x)

### 4.一个block

In [7]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head_dim = config.head_dim

        self.att = MultiHeadAttention(config)
        self.FFN = FeedForward(config)
        self.LN1 = nn.LayerNorm(config.hidden_dim)
        self.LN2 = nn.LayerNorm(config.hidden_dim)
    
    def forward(self,x):
        x = x + self.att(self.LN1(x))
        x = x + self.FFN(self.LN2(x))
        return x
        


### 5.完整的GPT

![GPT-2结构图](picture/GPT-2.webp)

In [8]:
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        # Toekn Embed
        self.token_embedding = nn.Embedding(config.vocab_size,config.hidden_dim)
        # Position Embed
        self.position_embedding = nn.Embedding(config.max_seq_len,config.hidden_dim)
        # LayerNorm_final
        self.LN_final = nn.LayerNorm(config.hidden_dim)
        # Linear
        self.lm_head = nn.Linear(config.hidden_dim,config.vocab_size,bias=False)
        # block
        # *是unpack - 把列表进行解包
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )

        # 让整个模型里所有子模块，都自动执行一遍 _init_weights 这个函数
        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
    
    def forward(self,x,target = None):
        batch_size,seq_len = x.size()
        
        # 1.输入:
        token_emb = self.token_embedding(x)
        # arange(seq_len) - 生成对应token的位置编号
        pos_emb = self.position_embedding(
            torch.arange(seq_len,device=x.device)
        )   
        x = token_emb + pos_emb

        # 2.block - Transformer架构
        x = self.blocks(x)

        # 3.LayerNorm
        x = self.LN_final(x)

        # 4.Linear
        logits = self.lm_head(x)

        if target is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
        
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # 如果序列太长，只取最后 max_seq_len 个token
            idx_cond = idx if idx.size(1) <= self.max_seq_len else idx[:, -self.max_seq_len:]
            # 获取预测
            logits, _ = self(idx_cond)
            # 只关注最后一个时间步的预测
            logits = logits[:, -1, :]  # becomes (B, vocab_size)
            # 应用softmax获取概率
            probs = F.softmax(logits, dim=-1)
            # 采样下一个token
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # 附加到序列上
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx


In [14]:
# 写一个 dataset，为了 Dataloader 准备
class MyDataset(Dataset):
    def __init__(self, path, block_size=512):
        # 我的数据在 /root/fs/mobvoi_seq_monkey_general_open_corpus.jsonl 中，
        # 读取前 1000 行
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size

        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"}
        )[0]

        import json

        self.encoded_data = []

        self.max_lines = 1000
        raw_data = []
        with open(path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= self.max_lines:
                    break
                try:
                    text = json.loads(line.strip())['text']
                    raw_data.append(text)
                except json.JSONDecodeError:
                    continue
                except Exception as e:
                    continue
        full_encoded = []
        for text in raw_data:
            encoded_text = self.enc.encode(text)
            full_encoded.extend(encoded_text + [self.eos_token])
        
        # 将长文本分割成训练样本
        for i in range(0, len(full_encoded), self.block_size):
            # 多取一个 Token 作为目标
            chunk = full_encoded[i:i+self.block_size+1]
            # 如果长度不够，用 eos_token 填充
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [self.eos_token] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)
    
    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

    def encode(self, text):
        """将文本编码为token IDs"""
        return self.enc.encode(text)

    def decode(self, ids):
        """将token IDs解码为文本"""
        return self.enc.decode(ids)

In [15]:

# train data
train_dataset = MyDataset('dataset\mobvoi_seq_monkey_general_open_corpus_1000.json')

# split traindataset to train and val
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.9, 0.1])

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)



e:\AI_Learning\.venv\Lib\site-packages\torch\utils\data\dataset.py:473: UserWarning: Length of split at index 0 is 0. This might result in an empty dataset.
  warnings.warn(
e:\AI_Learning\.venv\Lib\site-packages\torch\utils\data\dataset.py:473: UserWarning: Length of split at index 1 is 0. This might result in an empty dataset.
  warnings.warn(


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [9]:
model = GPT(GPTConfig())
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 打印模型一共有多少参数

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6} M")

Total parameters: 120.116736 M
